In [2]:
import json 
access_token_path = "/Users/alokbharadwaj/Desktop/hf_access.json"
with open(access_token_path, "r") as f:
    HF_INFERENCE_TOKEN = json.load(f)["token"]

In [3]:
from huggingface_hub import InferenceClient


In [4]:
client = InferenceClient(token=HF_INFERENCE_TOKEN)


In [5]:
output = client.chat.completions.create(
    model="meta-llama/Llama-3.1-8B-Instruct",
    messages=[
        {"role": "user", "content": "The capital of France is"}
    ],
    max_tokens=10,
    logprobs=True
)

print(output.choices[0].message.content)

The capital of France is Paris.


In [12]:
print(output.choices[0].logprobs)


ChatCompletionOutputLogprobs(content=[ChatCompletionOutputLogprob(logprob=-1.1056557893753052, token='Paris', top_logprobs=[], bytes=[80, 97, 114, 105, 115]), ChatCompletionOutputLogprob(logprob=-0.02729865349829197, token='.', top_logprobs=[], bytes=[46]), ChatCompletionOutputLogprob(logprob=-0.0069394768215715885, token='<|eot_id|>', top_logprobs=[], bytes=[60, 124, 101, 111, 116, 95, 105, 100, 124, 62])])


In [13]:
import math

for token in output.choices[0].logprobs.content:
    prob = math.exp(token.logprob)
    print(f"Token: {token.token!r:20} | logprob: {token.logprob:.3f} | prob: {prob:.3%}")

Token: 'Paris'              | logprob: -1.106 | prob: 33.099%
Token: '.'                  | logprob: -0.027 | prob: 97.307%
Token: '<|eot_id|>'         | logprob: -0.007 | prob: 99.308%


In [14]:
output = client.chat.completions.create(
    model="meta-llama/Llama-3.1-8B-Instruct",
    messages=[
        {"role": "user", "content": "The capital of France is"}
    ],
    max_tokens=5,
    logprobs=True,
    top_logprobs=5  # give me top 5 alternatives at each position
)

for token in output.choices[0].logprobs.content:
    print(f"\nChosen token: {token.token!r} ({math.exp(token.logprob):.3%})")
    print("Alternatives:")
    for alt in token.top_logprobs:
        print(f"  {alt.token!r:20} | prob: {math.exp(alt.logprob):.3%}")


Chosen token: 'The' (66.863%)
Alternatives:
  'The'                | prob: 66.863%
  'Paris'              | prob: 33.099%
  'France'             | prob: 0.026%
  'the'                | prob: 0.006%
  'Par'                | prob: 0.001%

Chosen token: ' capital' (99.998%)
Alternatives:
  ' capital'           | prob: 99.998%
  ' answer'            | prob: 0.001%
  ' city'              | prob: 0.000%
  ' Capital'           | prob: 0.000%
  'capital'            | prob: 0.000%

Chosen token: ' of' (100.000%)
Alternatives:
  ' of'                | prob: 100.000%
  ' city'              | prob: 0.000%
  ' is'                | prob: 0.000%
  ' and'               | prob: 0.000%
  ' '                  | prob: 0.000%

Chosen token: ' France' (100.000%)
Alternatives:
  ' France'            | prob: 100.000%
  'France'             | prob: 0.000%
  ' france'            | prob: 0.000%
  ' the'               | prob: 0.000%
  ' '                  | prob: 0.000%

Chosen token: ' is' (100.000%)
Alternativ

In [29]:
output = client.chat.completions.create(
    model="meta-llama/Llama-3.1-8B-Instruct",
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Answer in 3-4 sentences."},
        {"role": "user", "content": "What is a black hole?"}
    ],
    max_tokens=150,
    logprobs=True,
    top_logprobs=3
)

# Print the full response
print(output.choices[0].message.content)
print("---")

# Print logprob for every token in the response
for token in output.choices[0].logprobs.content:
    print(f"Token: {token.token!r:20} | logprob: {token.logprob:.3f} | prob: {math.exp(token.logprob):.3%}")

A black hole is a region in space where the gravitational pull is so strong that nothing, including light, can escape once it gets too close to the center. It is formed when a massive star collapses in on itself and its gravity becomes so intense that it warps the fabric of spacetime. The point of no return around a black hole is called the event horizon. The gravity of a black hole is so strong that it traps everything that crosses the event horizon.
---
Token: 'A'                  | logprob: -0.000 | prob: 100.000%
Token: ' black'             | logprob: -0.000 | prob: 100.000%
Token: ' hole'              | logprob: -0.000 | prob: 100.000%
Token: ' is'                | logprob: -0.000 | prob: 100.000%
Token: ' a'                 | logprob: -0.003 | prob: 99.692%
Token: ' region'            | logprob: -0.005 | prob: 99.501%
Token: ' in'                | logprob: -0.000 | prob: 99.958%
Token: ' space'             | logprob: -0.000 | prob: 99.992%
Token: ' where'             | logprob: -

In [30]:
def test_model(model_name):
    try:
        output = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": "What is the capital of France?"}],
            max_tokens=10,
            logprobs=True,
            top_logprobs=3
        )
        print(f"✅ {model_name}")
        print(f"   Response: {output.choices[0].message.content}")
        for token in output.choices[0].logprobs.content:
            print(f"   Token: {token.token!r:15} | logprob: {token.logprob:.3f}")
    except Exception as e:
        print(f"❌ {model_name}")
        print(f"   Error: {e}")

In [32]:
models = [
    "mistralai/Mistral-7B-Instruct-v0.3",
    "google/gemma-2-2b-it",
    "Qwen/Qwen2.5-7B-Instruct",
    "microsoft/Phi-3.5-mini-instruct",
    "moonshotai/Kimi-K2-Instruct-0905",
]

for model in models:
    test_model(model)
    print()

❌ mistralai/Mistral-7B-Instruct-v0.3
   Error: (Request ID: Root=1-69d63c4f-6efe9d7414368000438919d8;88c2f8d9-c919-4a02-96bd-79d1525916d7)

Bad request:
{'message': "The requested model 'mistralai/Mistral-7B-Instruct-v0.3' is not a chat model.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}

❌ google/gemma-2-2b-it
   Error: (Request ID: Root=1-69d63c4f-3f48b4d0612dbb633f2aef52;8aa98ce7-5cfb-4390-8745-328db88a7589)

Bad request:
{'message': "The requested model 'google/gemma-2-2b-it' is not supported by any provider you have enabled.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}

✅ Qwen/Qwen2.5-7B-Instruct
   Response: The capital of France is Paris.
❌ Qwen/Qwen2.5-7B-Instruct
   Error: 'NoneType' object is not iterable

❌ microsoft/Phi-3.5-mini-instruct
   Error: (Request ID: Root=1-69d63c50-372d0a625d03741c58546eef;9fd217d6-e880-4d38-bebf-a873c7c327d2)

Bad request:
{'message': "The requested model 'microsoft/Ph

In [33]:
completion = client.chat.completions.create(
    model="moonshotai/Kimi-K2-Instruct-0905",
    messages=[
        {
            "role": "user",
            "content": "Generate a poem about the sea."
        }
    ],
)


In [38]:
test_model("mistralai/Mistral-7B-Instruct-v0.3")

❌ mistralai/Mistral-7B-Instruct-v0.3
   Error: (Request ID: Root=1-69d63e7a-26715b437f5d75c02430dd19;d6c240d2-807d-488c-8560-73dab101ea12)

Bad request:
{'message': "The requested model 'mistralai/Mistral-7B-Instruct-v0.3' is not a chat model.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}


In [39]:
models = [
    "mistralai/Mistral-7B-Instruct-v0.1",
    "mistralai/Mistral-Nemo-Instruct-2407",
    "mistralai/Mistral-Small-Instruct-2409",
]

for model in models:
    test_model(model)
    print()

❌ mistralai/Mistral-7B-Instruct-v0.1
   Error: (Request ID: Root=1-69d63e8e-4391fa503e6cbbff512d70e6;484b7066-8621-4f3e-9a35-2b0a8833cfe3)

Bad request:
{'message': "The requested model 'mistralai/Mistral-7B-Instruct-v0.1' is not supported by any provider you have enabled.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}

❌ mistralai/Mistral-Nemo-Instruct-2407
   Error: (Request ID: Root=1-69d63e8f-5f15b3514212476f1796035d;150f4bff-7c69-4ac1-aff7-febc62308b4d)

Bad request:
{'message': "The requested model 'mistralai/Mistral-Nemo-Instruct-2407' is not a chat model.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}

❌ mistralai/Mistral-Small-Instruct-2409
   Error: (Request ID: Root=1-69d63e8f-40cf52622ddbe589540f66b3;07423487-f3bd-4bec-be5a-7725aa2a354f)

Bad request:
{'message': "The requested model 'mistralai/Mistral-Small-Instruct-2409' is not a chat model.", 'type': 'invalid_request_error', 'param': 'model', 'code

In [40]:
models = [
    "deepseek-ai/DeepSeek-R1",
    "Qwen/Qwen3-Coder-Next",
    "openai/gpt-oss-120b",
]

for model in models:
    test_model(model)
    print()

✅ deepseek-ai/DeepSeek-R1
   Response: 
   Token: '<think>'       | logprob: -0.000
   Token: '\n'            | logprob: 0.000
   Token: 'Okay'          | logprob: -0.016
   Token: ','             | logprob: -0.000
   Token: ' the'          | logprob: -0.015
   Token: ' user'         | logprob: 0.000
   Token: ' asked'        | logprob: -0.666
   Token: ' "'            | logprob: -1.467
   Token: 'What'          | logprob: -0.000
   Token: ' is'           | logprob: -0.000

✅ Qwen/Qwen3-Coder-Next
   Response: The capital of France is **Paris**.
❌ Qwen/Qwen3-Coder-Next
   Error: 'NoneType' object has no attribute 'content'

✅ openai/gpt-oss-120b
   Response: None
   Token: 'User'          | logprob: -1.165
   Token: ' asks'         | logprob: -0.010
   Token: ':'             | logprob: -0.530
   Token: ' "'            | logprob: -0.015
   Token: 'What'          | logprob: -0.000
   Token: ' is'           | logprob: 0.000
   Token: ' the'          | logprob: -0.000



In [41]:
output = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "What is the capital of France?"}],
    max_tokens=10,
    logprobs=True,
    top_logprobs=3
)

print(output)

ChatCompletionOutput(choices=[ChatCompletionOutputComplete(finish_reason='length', index=0, message=ChatCompletionOutputMessage(role='assistant', content=None, reasoning='The user asks: "What is', tool_call_id=None, tool_calls=None), logprobs=ChatCompletionOutputLogprobs(content=[ChatCompletionOutputLogprob(logprob=-0.4147226810455322, token='The', top_logprobs=[ChatCompletionOutputTopLogprob(logprob=-0.4147226810455322, token='The', bytes=[84, 104, 101]), ChatCompletionOutputTopLogprob(logprob=-1.1647226810455322, token='User', bytes=[85, 115, 101, 114]), ChatCompletionOutputTopLogprob(logprob=-3.9147226810455322, token='We', bytes=[87, 101])], bytes=[84, 104, 101]), ChatCompletionOutputLogprob(logprob=-0.00976228341460228, token=' user', top_logprobs=[ChatCompletionOutputTopLogprob(logprob=-0.00976228341460228, token=' user', bytes=[32, 117, 115, 101, 114]), ChatCompletionOutputTopLogprob(logprob=-4.759762287139893, token=' question', bytes=[32, 113, 117, 101, 115, 116, 105, 111, 110

In [10]:
def test_model(model_name):
    try:
        output = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": "What is the capital of Kazakhstan?"}],
            max_tokens=10,
            logprobs=True,
            top_logprobs=3
        )
        
        message = output.choices[0].message
        response_text = message.content if message.content is not None else message.reasoning
        
        print(f"✅ {model_name}")
        print(f"   Response: {response_text}")
        for token in output.choices[0].logprobs.content:
            print(f"   Token: {token.token!r:15} | logprob: {token.logprob:.3f}")
    except Exception as e:
        print(f"❌ {model_name}")
        print(f"   Error: {e}")

In [43]:
test_model("openai/gpt-oss-120b")

✅ openai/gpt-oss-120b
   Response: The user asks a straightforward factual question
   Token: 'The'           | logprob: -0.415
   Token: ' user'         | logprob: -0.010
   Token: ' asks'         | logprob: -0.009
   Token: ' a'            | logprob: -0.836
   Token: ' straightforward' | logprob: -1.413
   Token: ' factual'      | logprob: -1.071
   Token: ' question'     | logprob: -0.000


In [9]:
output = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "system", "content": "Reasoning: low"},
        {"role": "user", "content": "What is the capital of Kazakhstan?"}
    ],
    max_tokens=40,
    logprobs=True,
    top_logprobs=3
)

message = output.choices[0].message
response_text = message.content if message.content is not None else message.reasoning
print(response_text)

User asks: "What is the capital of Kazakhstan?" Answer: It's Astana (renamed to Nur-Sultan from 2019 to 2022?), but recently in Dec 


In [11]:
models = [
    "meta-llama/Llama-3.1-8B-Instruct",   # current - already working
    "meta-llama/Llama-3.2-3B-Instruct",   # smaller, newer
    "meta-llama/Llama-3.2-1B-Instruct",   # tiny
    "meta-llama/Meta-Llama-3-8B-Instruct", # previous generation
]

for model in models:
    test_model(model)
    print()

✅ meta-llama/Llama-3.1-8B-Instruct
   Response: The capital of Kazakhstan is Nur-Sultan (pre
   Token: 'The'           | logprob: -0.011
   Token: ' capital'      | logprob: -0.000
   Token: ' of'           | logprob: -0.000
   Token: ' Kazakhstan'   | logprob: -0.000
   Token: ' is'           | logprob: -0.000
   Token: ' Nur'          | logprob: -2.254
   Token: '-S'            | logprob: -0.000
   Token: 'ultan'         | logprob: -0.000
   Token: ' ('            | logprob: -0.856
   Token: 'pre'           | logprob: -0.385

❌ meta-llama/Llama-3.2-3B-Instruct
   Error: (Request ID: Root=1-69d64bd2-319742ca7962847569421895;0cfe6127-884f-4831-a2e9-05e50d40f696)

Bad request:
{'message': "The requested model 'meta-llama/Llama-3.2-3B-Instruct' is not supported by any provider you have enabled.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}

✅ meta-llama/Llama-3.2-1B-Instruct
   Response: The capital of Kazakhstan is Astana.
   Token: 'The'           